# NCS-O*NET 매핑 : Step 2 — 의미 유사도 기반 후보 생성
## 목표: NCS 능력단위(51개) ↔ O*NET 직업(1,016개) 임베딩 + 코사인 유사도 행렬 계산

## Cell 1: 라이브러리 import 및 경로 설정

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm
from deep_translator import GoogleTranslator
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings("ignore")

RAW_DIR       = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")
EMBED_DIR     = Path("../data/processed/embeddings")
OUTPUT_DIR    = Path("../outputs")
ONET_DIR      = RAW_DIR / "db_30_2_excel"

for d in [PROCESSED_DIR, EMBED_DIR, OUTPUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# 임베딩 모델 설정
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"  # 384-dim, 256 token max, CPU OK
TOP_K      = 20   # 능력단위당 추출할 O*NET 후보 수
BATCH_SIZE = 64   # O*NET 인코딩 배치 크기

print(f"MODEL  : {MODEL_NAME}")
print(f"TOP_K  : {TOP_K}")
print(f"경로 설정 완료")

## Cell 2: NCS 능력단위 레벨 텍스트 재구성 (중복 제거 + 클린 텍스트)

In [ ]:
ncs_raw = pd.read_excel(RAW_DIR / "NCS_DB.xlsx", engine="openpyxl").fillna("")

# 계층 컬럼 자동 탐지
def find_col(df, *keywords):
    """컬럼명에 모든 keyword가 포함된 첫 컬럼 반환."""
    for col in df.columns:
        if all(k in col for k in keywords):
            return col
    return None

COL_SUB    = find_col(ncs_raw, "소분류", "명")   # 소분류코드명
COL_DET    = find_col(ncs_raw, "세분류", "명")   # 세분류코드명
COL_UNIT   = find_col(ncs_raw, "능력단위명칭")   # 능력단위명칭
COL_ELEM   = find_col(ncs_raw, "능력단위요소명") # 능력단위요소명
COL_PERF   = find_col(ncs_raw, "수행준거") and next(
    (c for c in ncs_raw.columns if "수행준거" in c and "번호" not in c), None
)
COL_KSA_NM = find_col(ncs_raw, "지식기술태도코드명")  # 지식/기술/태도 구분
COL_KSA    = find_col(ncs_raw, "지식기술태도의의")    # KSA 내용

print(f"소분류코드명   : {COL_SUB}")
print(f"세분류코드명   : {COL_DET}")
print(f"능력단위명칭   : {COL_UNIT}")
print(f"능력단위요소명 : {COL_ELEM}")
print(f"수행준거       : {COL_PERF}")
print(f"지식기술태도코드명 : {COL_KSA_NM}")
print(f"지식기술태도의의   : {COL_KSA}")

def unique_join(series, sep=". "):
    """중복 제거 후 문자열 연결 (빈 값 제외)."""
    return sep.join(dict.fromkeys(v for v in series.astype(str) if v.strip()))

# 능력단위(UNIT) 레벨로 그룹핑하여 클린 텍스트 생성
records = []
group_cols = [COL_SUB, COL_DET, COL_UNIT]

for keys, grp in ncs_raw.groupby(group_cols):
    sub_nm, det_nm, unit_nm = keys

    # KSA를 유형별로 분리
    ksa_dict = {ktype: [] for ktype in ["지식", "기술", "태도"]}
    for _, row in grp.iterrows():
        ktype = str(row[COL_KSA_NM]).strip()
        val   = str(row[COL_KSA]).strip()
        if ktype in ksa_dict and val:
            ksa_dict[ktype].append(val)

    def dedup(lst):
        return ". ".join(dict.fromkeys(lst))

    # O*NET Task 구조와 유사하게 구성:
    # "직무명. 세분류. [능력단위요소들]. 수행기준. 지식: ... 기술: ... 태도: ..."
    elem_text  = unique_join(grp[COL_ELEM])
    perf_text  = unique_join(grp[COL_PERF]) if COL_PERF else ""
    know_text  = dedup(ksa_dict["지식"])
    skill_text = dedup(ksa_dict["기술"])
    atti_text  = dedup(ksa_dict["태도"])

    parts = [
        f"{unit_nm} ({det_nm}, {sub_nm}).",
        f"Tasks: {elem_text}.",
        f"Performance criteria: {perf_text}." if perf_text else "",
        f"Knowledge: {know_text}."  if know_text  else "",
        f"Skills: {skill_text}."    if skill_text else "",
        f"Attitudes: {atti_text}."  if atti_text  else "",
    ]
    clean_text = " ".join(p for p in parts if p)

    records.append({
        "소분류코드명"  : sub_nm,
        "세분류코드명"  : det_nm,
        "능력단위명칭"  : unit_nm,
        "clean_text_ko" : clean_text,
        "text_len_ko"   : len(clean_text),
    })

ncs_units = pd.DataFrame(records)
print(f"\n능력단위 수: {len(ncs_units)}")
print(f"\n세분류별 능력단위 수:")
print(ncs_units.groupby("세분류코드명")["능력단위명칭"].count().to_string())
print(f"\n텍스트 길이 통계:")
print(ncs_units["text_len_ko"].describe().round(0).to_string())
print(f"\n샘플 텍스트 (첫 300자):\n{ncs_units['clean_text_ko'].iloc[0][:300]}")

## Cell 3: NCS 능력단위 텍스트 영어 번역

In [ ]:
CACHE_PATH  = PROCESSED_DIR / "ncs_units_translated.csv"
CHUNK_SIZE  = 500   # GoogleTranslator 안전 청크

def translate_text(text: str, src: str = "ko", tgt: str = "en") -> str:
    """긴 텍스트를 청크 단위로 번역."""
    if not text or not text.strip():
        return text
    translator = GoogleTranslator(source=src, target=tgt)
    chunks = [text[i : i + CHUNK_SIZE] for i in range(0, len(text), CHUNK_SIZE)]
    results = []
    for chunk in chunks:
        try:
            results.append(translator.translate(chunk))
        except Exception as e:
            print(f"  [경고] 번역 실패 (원문 유지): {e}")
            results.append(chunk)
    return " ".join(results)

# 캐시가 이미 있으면 재번역하지 않음
if CACHE_PATH.exists():
    ncs_units = pd.read_csv(CACHE_PATH, encoding="utf-8-sig")
    print(f"캐시 로드: {CACHE_PATH}")
    print(f"shape: {ncs_units.shape}")
else:
    print(f"번역 시작 (능력단위 {len(ncs_units)}개)...")
    translated = []
    for i, row in tqdm(ncs_units.iterrows(), total=len(ncs_units), desc="번역"):
        en = translate_text(row["clean_text_ko"])
        translated.append(en)
        if (i + 1) % 10 == 0:
            print(f"  [{i+1}/{len(ncs_units)}] 샘플: {en[:80]}...")

    ncs_units["clean_text_en"] = translated
    ncs_units["text_len_en"]   = ncs_units["clean_text_en"].str.len()
    ncs_units.to_csv(CACHE_PATH, index=False, encoding="utf-8-sig")
    print(f"\n저장 완료: {CACHE_PATH}")

print(f"\n번역 텍스트 길이 통계:")
print(ncs_units["text_len_en"].describe().round(0).to_string())
print(f"\n[샘플 번역] {ncs_units['능력단위명칭'].iloc[0]}")
print(ncs_units["clean_text_en"].iloc[0][:400])

## Cell 4: 임베딩 모델 로드 + NCS 임베딩 생성

In [ ]:
print(f"모델 로드 중: {MODEL_NAME}")
print("(최초 실행 시 모델 다운로드 — 약 80MB, 수십 초 소요)")

model = SentenceTransformer(MODEL_NAME)

print(f"  Max sequence length : {model.max_seq_length} tokens")
print(f"  Embedding dimension : {model.get_sentence_embedding_dimension()}")

# ── NCS 능력단위 임베딩 ─────────────────────────────────────────────
NCS_EMBED_PATH = EMBED_DIR / "ncs_unit_embeddings.npy"

if NCS_EMBED_PATH.exists():
    ncs_embeddings = np.load(NCS_EMBED_PATH)
    print(f"\nNCS 임베딩 캐시 로드: {NCS_EMBED_PATH}")
else:
    ncs_texts = ncs_units["clean_text_en"].tolist()
    print(f"\nNCS 임베딩 생성 중 ({len(ncs_texts)}개)...")
    ncs_embeddings = model.encode(
        ncs_texts,
        batch_size=BATCH_SIZE,
        show_progress_bar=True,
        normalize_embeddings=True,   # L2 정규화 → cosine = dot product
    )
    np.save(NCS_EMBED_PATH, ncs_embeddings)
    print(f"저장 완료: {NCS_EMBED_PATH}")

print(f"\nncs_embeddings shape : {ncs_embeddings.shape}")
print(f"벡터 norm 확인 (정규화 체크): {np.linalg.norm(ncs_embeddings[0]):.4f}")

## Cell 5: O*NET 임베딩 생성

In [ ]:
onet = pd.read_csv(PROCESSED_DIR / "onet_processed.csv", encoding="utf-8-sig")
onet["final_text"] = onet["final_text"].fillna("").astype(str)

print(f"O*NET 직업 수  : {len(onet)}")
print(f"final_text 길이: 평균 {onet['final_text'].str.len().mean():.0f}자 / "
      f"최대 {onet['final_text'].str.len().max()}자")
print(f"모델 max_seq   : {model.max_seq_length} tokens "
      f"(≈{model.max_seq_length * 4}자 — 초과분은 자동 truncation)")

# ── O*NET 임베딩 ───────────────────────────────────────────────────
ONET_EMBED_PATH = EMBED_DIR / "onet_embeddings.npy"

if ONET_EMBED_PATH.exists():
    onet_embeddings = np.load(ONET_EMBED_PATH)
    print(f"\nO*NET 임베딩 캐시 로드: {ONET_EMBED_PATH}")
else:
    onet_texts = onet["final_text"].tolist()
    print(f"\nO*NET 임베딩 생성 중 ({len(onet_texts)}개, batch={BATCH_SIZE})...")
    print("CPU 기준 약 2~5분 소요 예상")
    onet_embeddings = model.encode(
        onet_texts,
        batch_size=BATCH_SIZE,
        show_progress_bar=True,
        normalize_embeddings=True,
    )
    np.save(ONET_EMBED_PATH, onet_embeddings)
    print(f"\n저장 완료: {ONET_EMBED_PATH}")

print(f"\nonet_embeddings shape : {onet_embeddings.shape}")
print(f"벡터 norm 확인       : {np.linalg.norm(onet_embeddings[0]):.4f}")

## Cell 6: 코사인 유사도 행렬 계산 (51 × 1,016)

In [ ]:
# normalize_embeddings=True이므로 cosine_similarity = dot product
# shape: (51, 1016)
sim_matrix = cosine_similarity(ncs_embeddings, onet_embeddings)

print(f"유사도 행렬 shape : {sim_matrix.shape}  (NCS능력단위 × O*NET직업)")
print(f"값 범위           : {sim_matrix.min():.4f} ~ {sim_matrix.max():.4f}")
print(f"전체 평균 유사도  : {sim_matrix.mean():.4f}")

# 능력단위별 최고 유사도 O*NET 직업 미리보기
print(f"\n[능력단위별 Top-1 O*NET 매핑 미리보기]")
print(f"{'NCS 능력단위명':<35} {'Top-1 O*NET 직업':<40} {'유사도':>6}")
print("-" * 85)
for i, row in ncs_units.iterrows():
    best_idx   = sim_matrix[i].argmax()
    best_score = sim_matrix[i, best_idx]
    best_title = onet.iloc[best_idx]["Title"]
    unit_name  = row["능력단위명칭"]
    print(f"{unit_name[:34]:<35} {best_title[:39]:<40} {best_score:.4f}")

# 전체 행렬 저장 (npy)
SIM_MATRIX_PATH = EMBED_DIR / "sim_matrix_unit_onet.npy"
np.save(SIM_MATRIX_PATH, sim_matrix)
print(f"\n유사도 행렬 저장 완료: {SIM_MATRIX_PATH}")

## Cell 7: Top-K 후보 추출 — 능력단위 / 세분류 / 소분류 3단계

In [ ]:
def build_topk_df(sim_rows, meta_df, onet_df, k, groupby_col):
    """
    sim_rows : 2D ndarray (N x 1016)
    meta_df  : NCS 메타 DataFrame (sim_rows와 행 순서 일치)
    onet_df  : O*NET DataFrame
    k        : 추출할 후보 수
    groupby_col : 그룹핑 기준 컬럼명
    반환 : 롱 포맷 DataFrame
    """
    records = []
    # 그룹별 행 인덱스 수집
    groups = meta_df.groupby(groupby_col).apply(lambda x: x.index.tolist())

    for group_name, idx_list in groups.items():
        # 해당 그룹 능력단위들의 유사도 행 평균 → 집계 점수
        agg_scores = sim_rows[idx_list].mean(axis=0)  # (1016,)
        top_indices = np.argsort(agg_scores)[::-1][:k]

        for rank, onet_idx in enumerate(top_indices, start=1):
            onet_row = onet_df.iloc[onet_idx]
            records.append({
                groupby_col          : group_name,
                "rank"               : rank,
                "O*NET-SOC Code"     : onet_row["O*NET-SOC Code"],
                "O*NET Title"        : onet_row["Title"],
                "cosine_similarity"  : round(float(agg_scores[onet_idx]), 4),
                "O*NET Description"  : str(onet_row.get("Description", ""))[:200],
            })
    return pd.DataFrame(records)


# ── Level 1: 능력단위별 Top-K ─────────────────────────────────────
topk_unit = []
for i, row in ncs_units.iterrows():
    scores      = sim_matrix[i]           # (1016,)
    top_indices = np.argsort(scores)[::-1][:TOP_K]
    for rank, onet_idx in enumerate(top_indices, start=1):
        onet_row = onet.iloc[onet_idx]
        topk_unit.append({
            "소분류코드명"       : row["소분류코드명"],
            "세분류코드명"       : row["세분류코드명"],
            "능력단위명칭"       : row["능력단위명칭"],
            "rank"               : rank,
            "O*NET-SOC Code"     : onet_row["O*NET-SOC Code"],
            "O*NET Title"        : onet_row["Title"],
            "cosine_similarity"  : round(float(scores[onet_idx]), 4),
            "O*NET Description"  : str(onet_row.get("Description", ""))[:200],
        })
topk_unit_df = pd.DataFrame(topk_unit)

# ── Level 2: 세분류별 Top-K (능력단위 유사도 평균) ───────────────
topk_det_df = build_topk_df(sim_matrix, ncs_units, onet, TOP_K, "세분류코드명")

# ── Level 3: 소분류별 Top-K (전체 능력단위 유사도 평균) ──────────
topk_sub_df = build_topk_df(sim_matrix, ncs_units, onet, TOP_K, "소분류코드명")

print(f"능력단위별 후보 : {len(topk_unit_df)}행  ({len(ncs_units)}능력단위 × Top-{TOP_K})")
print(f"세분류별 후보   : {len(topk_det_df)}행  ({ncs_units['세분류코드명'].nunique()}세분류 × Top-{TOP_K})")
print(f"소분류별 후보   : {len(topk_sub_df)}행  ({ncs_units['소분류코드명'].nunique()}소분류 × Top-{TOP_K})")

print(f"\n[세분류별 Top-3 미리보기]")
print(topk_det_df[topk_det_df["rank"] <= 3]
      [["세분류코드명", "rank", "O*NET Title", "cosine_similarity"]]
      .to_string(index=False))

## Cell 8: 결과 저장 — CSV + Excel (시트 분리)

In [ ]:
# ── CSV 저장 ──────────────────────────────────────────────────────
CSV_UNIT = OUTPUT_DIR / "mapping_candidates_unit.csv"
CSV_DET  = OUTPUT_DIR / "mapping_candidates_detail.csv"
CSV_SUB  = OUTPUT_DIR / "mapping_candidates_sub.csv"

topk_unit_df.to_csv(CSV_UNIT, index=False, encoding="utf-8-sig")
topk_det_df.to_csv(CSV_DET,  index=False, encoding="utf-8-sig")
topk_sub_df.to_csv(CSV_SUB,  index=False, encoding="utf-8-sig")

# ── Excel 멀티시트 저장 ───────────────────────────────────────────
EXCEL_PATH = OUTPUT_DIR / "ncs_onet_mapping_candidates.xlsx"

with pd.ExcelWriter(EXCEL_PATH, engine="openpyxl") as writer:
    # 시트1: 소분류 수준
    topk_sub_df.to_excel(writer, sheet_name="소분류_Top20", index=False)

    # 시트2: 세분류 수준
    topk_det_df.to_excel(writer, sheet_name="세분류_Top20", index=False)

    # 시트3~7: 세분류별 개별 시트
    for det_name, grp in topk_unit_df.groupby("세분류코드명"):
        sheet_name = det_name[:25]  # Excel 시트명 31자 제한
        grp.to_excel(writer, sheet_name=sheet_name, index=False)

    # 시트8: 능력단위 전체
    topk_unit_df.to_excel(writer, sheet_name="능력단위_전체_Top20", index=False)

print("저장 완료:")
for p in [CSV_UNIT, CSV_DET, CSV_SUB, EXCEL_PATH]:
    size_kb = p.stat().st_size / 1024
    print(f"  {p.name:<50} ({size_kb:,.1f} KB)")

## Cell 9: 최종 요약

In [ ]:
print("=" * 70)
print("  Step 2 완료 — 의미 유사도 기반 O*NET 후보 생성 요약")
print("=" * 70)

print(f"\n[임베딩 정보]")
print(f"  모델         : {MODEL_NAME}")
print(f"  NCS 임베딩   : {ncs_embeddings.shape}  (능력단위 × 384dim)")
print(f"  O*NET 임베딩 : {onet_embeddings.shape}  (직업 × 384dim)")
print(f"  유사도 행렬  : {sim_matrix.shape}  (min={sim_matrix.min():.3f}, max={sim_matrix.max():.3f})")

print(f"\n[세분류별 Top-1 매핑 결과]")
print(f"{'세분류':<22} {'O*NET Top-1 직업':<40} {'유사도':>6}")
print("-" * 72)
for _, row in topk_det_df[topk_det_df["rank"] == 1].iterrows():
    print(f"{row['세분류코드명'][:21]:<22} {row['O*NET Title'][:39]:<40} {row['cosine_similarity']:>6.4f}")

print(f"\n[소분류별 Top-1 매핑 결과]")
print(f"{'소분류':<18} {'O*NET Top-1 직업':<40} {'유사도':>6}")
print("-" * 68)
for _, row in topk_sub_df[topk_sub_df["rank"] == 1].iterrows():
    print(f"{row['소분류코드명'][:17]:<18} {row['O*NET Title'][:39]:<40} {row['cosine_similarity']:>6.4f}")

print(f"\n[저장된 파일]")
all_outputs = list(OUTPUT_DIR.glob("*")) + list(EMBED_DIR.glob("*"))
for p in sorted(all_outputs):
    size = p.stat().st_size
    unit = "KB" if size < 1_000_000 else "MB"
    val  = size / 1024 if size < 1_000_000 else size / 1_048_576
    print(f"  {p.relative_to(Path('../'))}  ({val:.1f} {unit})")

print(f"\n다음 단계 : Step 3 — 결과 검증 및 시각화")
print("  notebooks/03_visualization.ipynb")
print("=" * 70)